# Web Scraping Kaggle — Fluxo em 6 Células

1. Imports e bibliotecas  
2. URLs e requisição (HTML original do iframe)  
3. BeautifulSoup  
4. find_all() — extração  
5. Armazenar/manipular dados  
6. Salvar em SQL (banco infinito, pronto para ML)

## 1. Imports e bibliotecas

Inclui: `requests`, `BeautifulSoup`, `json`, `time`, `datetime`, Selenium (`webdriver`, `By`, `WebDriverWait`, `EC`, `Options`), `pandas`, `numpy`, `os`, `Path`, `logging`, `typing`, `subprocess`, `sys`.

In [1]:
# ========== CÉLULA 1: IMPORTS E BIBLIOTECAS (conforme celula 0.1) ==========
import requests
from bs4 import BeautifulSoup
import json
import time
from datetime import datetime

# Selenium — para carregar páginas com JS e obter HTML real (incl. iframe)
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options

# Processamento de dados
import pandas as pd
import numpy as np
import sqlite3

# Utilitários
import os
import sys
import subprocess
from pathlib import Path
import logging
from typing import List, Dict, Optional

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
import warnings
warnings.filterwarnings('ignore')

print("✓ Bibliotecas carregadas (requests, BeautifulSoup, Selenium, pandas, numpy, Path, logging, typing, subprocess, sys).")


✓ Bibliotecas carregadas (requests, BeautifulSoup, Selenium, pandas, numpy, Path, logging, typing, subprocess, sys).


## 2. URLs — Configuração e descoberta com Selenium

Define LISTING_URLS, SEARCH_URLS, DIRECT_NOTEBOOK_URLS, WRITEUP_URLS; funções de dedupe, normalização e extração de links; **fetch_with_selenium** (HTML com JS); **is_listing_page** e **extract_notebook_urls_from_listing**. Ao executar, roda a **ETAPA 1: descoberta e mapeamento** (total de notebooks descobertos ~249).

In [13]:
# CONFIGURAÇÃO: URLS + LIMITES PARA SCRAPING COM SELENIUM
LISTING_URLS = [
    "https://www.kaggle.com/code?sortBy=relevance&tab=hot&page=2",
    "https://www.kaggle.com/code?sortBy=relevance&tab=hot&page=3",
    "https://www.kaggle.com/code?sortBy=relevance&tab=hot&page=4",
    "https://www.kaggle.com/code?sortBy=relevance&tab=hot&page=5",
    "https://www.kaggle.com/code?sortBy=relevance&tab=hot&page=6",
    "https://www.kaggle.com/code?sortBy=relevance&tab=hot&page=7",
    "https://www.kaggle.com/code?sortBy=relevance&tab=hot&page=8",
    "https://www.kaggle.com/code?sortBy=relevance&tab=hot&page=9",
    "https://www.kaggle.com/code?sortBy=relevance&searchQuery=AIMO+Progress+Prize+1+Kaggle+notebooks",
    "https://www.kaggle.com/code?sortBy=relevance&searchQuery=AIMO+Progress+Prize+1+Kaggle+notebooks&page=2",
    "https://www.kaggle.com/code?searchQuery=Project+Numina+AIMO"
]
SEARCH_URLS = [
    "https://www.kaggle.com/search?q=AI+Mathematical+Olympiad+in%3Anotebooks",
    "https://www.kaggle.com/search?q=GSM8K+in%3Anotebooks"
]
DIRECT_NOTEBOOK_URLS = [
    "https://www.kaggle.com/code/artemgoncarov/aimo-2025-qwq-32b-solution",
    "https://www.kaggle.com/code/nayjest/bl-apiv2-microcore-chain-frame-1-pass-score-9",
    "https://www.kaggle.com/code/bsmit1659/aimo-vllm-accelerated-tot-sc-deepseekmath",
    "https://www.kaggle.com/code/anrenk/aimo-llm-class-deepseek",
    "https://www.kaggle.com/code/sorenravn/aimo-2-4th-place",
    "https://www.kaggle.com/code/julian3833/aimo2-starter-llm-code-baseline-lb-2",
    "https://www.kaggle.com/code/mpwolke/gemma-what-s-the-remainder-h100-2m12s",
    "https://www.kaggle.com/code/dschettler8845/aimo-let-s-learn-together",
    "https://www.kaggle.com/code/mbmmurad/lb-20-qwq-32b-preview-optimized-inference",
    "https://www.kaggle.com/code/rohanrk1813/march-mania",
    "https://www.kaggle.com/code/masanakashima/aimo3-initial-pipeline-inference-server-v2",
    "https://www.kaggle.com/code/runningpoppy/aimo-3-submission-demo",
    "https://www.kaggle.com/code/yashwanthnadella/fine-tune-llama3-8b-for-gsm8k"
]
WRITEUP_URLS = [
    "https://www.kaggle.com/competitions/ai-mathematical-olympiad-prize/writeups/cmu-math-2nd-place-solution-all-code-and-datasets-",
    "https://www.kaggle.com/competitions/ai-mathematical-olympiad-prize/writeups/after-exams-3rd-place-solution",
    "https://www.kaggle.com/competitions/ai-mathematical-olympiad-prize/writeups/codeinter-4th-place-solution",
    "https://www.kaggle.com/competitions/ai-mathematical-olympiad-prize/writeups/the-ai-of-pi-aimo-36th-place-solution",
    "https://www.kaggle.com/competitions/ai-mathematical-olympiad-prize/writeups/hayatofujihara-41st-place-solution",
    "https://www.kaggle.com/competitions/ai-mathematical-olympiad-progress-prize-2/writeups/imagination-research-2nd-place-solution-team-imagi",
    "https://www.kaggle.com/competitions/ai-mathematical-olympiad-progress-prize-2/writeups/aliev-3rd-place-solution-report",
    "https://www.kaggle.com/competitions/ai-mathematical-olympiad-progress-prize-2/writeups/sravn-4th-place-solution",
    "https://www.kaggle.com/competitions/ai-mathematical-olympiad-progress-prize-2/writeups/usernam-5th-place-solution",
    "https://www.kaggle.com/competitions/ai-mathematical-olympiad-progress-prize-2/writeups/tascj-7th-place-solution-pure-luck",
    "https://www.kaggle.com/competitions/ai-mathematical-olympiad-progress-prize-2/writeups/mpware-8th-place-solution-lb28-takeway",
    "https://www.kaggle.com/competitions/ai-mathematical-olympiad-progress-prize-2/writeups/fast-math-r1-14b-lb-pub-29-pvt-28-enhancing-the-ef",
    "https://www.kaggle.com/competitions/ai-mathematical-olympiad-progress-prize-2/writeups/farsail-11th-place-solution",
    "https://www.kaggle.com/competitions/ai-mathematical-olympiad-progress-prize-2/writeups/ippeiogawa-private27-public28-solution-for-17th-to",
    "https://www.kaggle.com/competitions/ai-mathematical-olympiad-progress-prize-2/writeups/arek-paterek-20th-place-solution-generate-lots-of-",
    "https://www.kaggle.com/competitions/ai-mathematical-olympiad-progress-prize-2/writeups/jk-piece-21st-place-solution",
    "https://www.kaggle.com/competitions/ai-mathematical-olympiad-progress-prize-2/writeups/nemoskills-1st-place-solution-nemoskills",
    "https://www.kaggle.com/competitions/llm-zoomcamp-2024-competition/writeups/vladkha-one-of-top-scoring-solutions-oss-model-gpt",
    "https://www.kaggle.com/competitions/ai-mathematical-olympiad-prize/writeups/numina-numina-1st-place-solution"
]
SEED_URLS = LISTING_URLS + SEARCH_URLS
MAX_SEED_URLS = None
MAX_NOTEBOOKS_TO_PROCESS = 999
DEBUG_PRETTIFY_CHARS = 2000

def dedupe_urls(urls):
    seen = set()
    unique = []
    for url in urls:
        if url not in seen:
            seen.add(url)
            unique.append(url)
    return unique

def normalize_url(href, base="https://www.kaggle.com"):
    if not href:
        return None
    if href.startswith("http"):
        return href
    if href.startswith("/"):
        return f"{base}{href}"
    return f"{base}/{href}"

def extract_links_from_html(html):
    soup = BeautifulSoup(html, 'html.parser')
    notebook_links, writeup_links, pagination_links = [], [], []
    for link in soup.find_all('a', href=True):
        href = link.get('href')
        full_url = normalize_url(href)
        if not full_url:
            continue
        href_lower = full_url.lower()
        if '/code/' in href_lower:
            notebook_links.append(full_url)
        if '/writeups/' in href_lower:
            writeup_links.append(full_url)
        if 'page=' in href_lower or '/search?' in href_lower:
            pagination_links.append(full_url)
    return dedupe_urls(notebook_links), dedupe_urls(writeup_links), dedupe_urls(pagination_links)

def is_listing_page(soup):
    try:
        if soup.select_one('#notebook-container'):
            return False
        code_links = soup.find_all('a', href=lambda x: x and '/code/' in x.lower())
        if len(code_links) >= 5:
            return True
        pagination = soup.find_all('a', class_=lambda x: x and ('next' in str(x).lower() or 'page' in str(x).lower()))
        if pagination:
            return True
        return False
    except Exception as e:
        logging.warning(f"Erro ao detectar tipo de página: {e}")
        return False

def extract_notebook_urls_from_listing(html_content):
    try:
        soup = html_content if hasattr(html_content, 'find_all') else BeautifulSoup(html_content, 'html.parser')
        all_links = soup.find_all('a', href=lambda x: x and '/code/' in x.lower())
        notebook_urls = []
        for link in all_links:
            href = link.get('href')
            if not href:
                continue
            full_url = normalize_url(href)
            if not full_url:
                continue
            for suffix in ['/comments', '/output', '/logs', '/versions', '/settings', '/data']:
                if full_url.endswith(suffix):
                    full_url = full_url.replace(suffix, '')
            if '/code/' in full_url and full_url not in notebook_urls:
                notebook_urls.append(full_url)
        return notebook_urls
    except Exception as e:
        logging.error(f"Erro ao extrair URLs de listagem: {e}")
        return []

def fetch_with_selenium(url, wait_time=15, return_cookies=False):
    """Obtém HTML da página com Selenium (conteúdo após JS; contorna reCAPTCHA se resolver manualmente)."""
    try:
        options = Options()
        options.add_argument('--start-maximized')
        options.add_argument('--disable-blink-features=AutomationControlled')
        options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')
        driver = webdriver.Chrome(options=options)
        driver.get(url)
        time.sleep(3)
        try:
            recaptcha = driver.find_element(By.CLASS_NAME, 'g-recaptcha')
            print("⚠️ reCAPTCHA detectado. Complete no navegador. Aguardando 60s...")
            WebDriverWait(driver, 60).until(EC.invisibility_of_element_located((By.CLASS_NAME, 'g-recaptcha')))
        except Exception:
            pass
        time.sleep(2)
        html_content = driver.page_source
        if return_cookies:
            out = {"html": html_content, "cookies": driver.get_cookies(), "user_agent": driver.execute_script("return navigator.userAgent;")}
            driver.quit()
            return out
        driver.quit()
        return html_content
    except Exception as e:
        print(f"❌ Erro Selenium: {e}")
        try:
            driver.quit()
        except Exception:
            pass
        return None

def fetch_html_with_selenium(url):
    return fetch_with_selenium(url)

DISCOVERED_NOTEBOOK_URLS = []
DISCOVERED_WRITEUP_URLS = []
DISCOVERED_PAGINATION_URLS = []

print("✓ Configuração de URLs e funções (fetch_with_selenium, is_listing_page, extract_notebook_urls_from_listing) carregada.")

# EXECUÇÃO ETAPA 1: DESCOBERTA & MAPEAMENTO COM SELENIUM
try:
    seed_urls = SEED_URLS if MAX_SEED_URLS is None else SEED_URLS[:MAX_SEED_URLS]
    print("\n" + "="*70)
    print("🔬 ETAPA 1: DESCOBERTA & MAPEAMENTO (COM SUPORTE A LISTAGENS)")
    print("="*70)
    for idx, test_url in enumerate(seed_urls, 1):
        print(f"[{idx}/{len(seed_urls)}] {test_url[:60]}...")
        html_content = fetch_html_with_selenium(test_url)
        if not html_content:
            continue
        soup = BeautifulSoup(html_content, 'html.parser')
        is_listing = is_listing_page(soup)
        if is_listing:
            notebook_urls = extract_notebook_urls_from_listing(html_content)
            DISCOVERED_NOTEBOOK_URLS.extend(notebook_urls)
        else:
            notebook_links, writeup_links, pagination_links = extract_links_from_html(html_content)
            DISCOVERED_NOTEBOOK_URLS.extend(notebook_links)
            DISCOVERED_WRITEUP_URLS.extend(writeup_links)
            DISCOVERED_PAGINATION_URLS.extend(pagination_links)
        time.sleep(1)
    DISCOVERED_NOTEBOOK_URLS[:] = dedupe_urls(DISCOVERED_NOTEBOOK_URLS)
    DISCOVERED_WRITEUP_URLS[:] = dedupe_urls(DISCOVERED_WRITEUP_URLS)
    DISCOVERED_PAGINATION_URLS[:] = dedupe_urls(DISCOVERED_PAGINATION_URLS)
    print("\n✅ ETAPA 1 CONCLUÍDA")
    print(f"  • Total notebooks descobertos: {len(DISCOVERED_NOTEBOOK_URLS)}")
    print(f"  • Total writeups: {len(DISCOVERED_WRITEUP_URLS)}")
    for i, url in enumerate(DISCOVERED_NOTEBOOK_URLS[:], 1):
        print(f"    {i}. {url}")
except Exception as e:
    print(f"❌ Erro na ETAPA 1: {e}")
    import traceback
    traceback.print_exc()

✓ Configuração de URLs e funções (fetch_with_selenium, is_listing_page, extract_notebook_urls_from_listing) carregada.

🔬 ETAPA 1: DESCOBERTA & MAPEAMENTO (COM SUPORTE A LISTAGENS)
[1/13] https://www.kaggle.com/code?sortBy=relevance&tab=hot&page=2...
[2/13] https://www.kaggle.com/code?sortBy=relevance&tab=hot&page=3...
[3/13] https://www.kaggle.com/code?sortBy=relevance&tab=hot&page=4...
[4/13] https://www.kaggle.com/code?sortBy=relevance&tab=hot&page=5...
[5/13] https://www.kaggle.com/code?sortBy=relevance&tab=hot&page=6...
[6/13] https://www.kaggle.com/code?sortBy=relevance&tab=hot&page=7...
[7/13] https://www.kaggle.com/code?sortBy=relevance&tab=hot&page=8...
[8/13] https://www.kaggle.com/code?sortBy=relevance&tab=hot&page=9...
[9/13] https://www.kaggle.com/code?sortBy=relevance&searchQuery=AIM...
[10/13] https://www.kaggle.com/code?sortBy=relevance&searchQuery=AIM...
[11/13] https://www.kaggle.com/code?searchQuery=Project+Numina+AIMO...
[12/13] https://www.kaggle.com/search?q=AI+Ma

## 3. Baixar HTML real (iframe)

Usar **Selenium** para abrir cada URL de notebook descoberta (ex.: os ~249), localizar o **iframe** `#rendered-kernel-content`, obter a URL do iframe e baixar o **HTML original do iframe** (conteúdo real do notebook). Salvar cada HTML em `results/` para uso na célula 4 (find_all).  
**OBS (celula 0.1):** O código deve carregar todos os notebooks descobertos e adaptar o fluxo para os 249 HTMLs reais do iframe; comentários indicam o que falta implementar (ex.: loop em DISCOVERED_NOTEBOOK_URLS, rate limit, salvamento por notebook).

In [14]:
# Baixar HTML real do iframe para cada notebook descoberto (ex.: 249).
# Requer: Selenium (célula 1), DISCOVERED_NOTEBOOK_URLS (célula 2).

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
# Limite opcional para teste; use None para processar todos (249)
MAX_NOTEBOOKS_TO_DOWNLOAD = 255  # Ex.: 5 para teste; depois trocar para MAX_NOTEBOOKS_TO_PROCESS ou None

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
    'Referer': 'https://www.kaggle.com/'
}
session = requests.Session()
session.headers.update(headers)

# Lista de URLs alvo: notebooks descobertos na ETAPA 1 (ou DIRECT_NOTEBOOK_URLS se ainda não rodou a descoberta)
urls_to_fetch = (DISCOVERED_NOTEBOOK_URLS or DIRECT_NOTEBOOK_URLS)[:MAX_NOTEBOOKS_TO_DOWNLOAD or 999]
saved_paths = []

for i, page_url in enumerate(urls_to_fetch, 1):
    print(f"[{i}/{len(urls_to_fetch)}] {page_url}")
    try:
        # 1) Obter HTML da página do notebook (com Selenium para JS; ou requests se não houver recaptcha)
        html_pagina = fetch_html_with_selenium(page_url) or ""
        if not html_pagina:
            continue
        soup_pagina = BeautifulSoup(html_pagina, 'html.parser')
        iframe = soup_pagina.find('iframe', id='rendered-kernel-content')
        if not iframe or not iframe.get('src'):
            # Fallback: salvar HTML da página
            safe_name = page_url.rstrip('/').split('/')[-1] or 'notebook'
            out_path = RESULTS_DIR / f"{safe_name}_page.html"
            out_path.write_text(html_pagina, encoding='utf-8')
            saved_paths.append(out_path)
            continue
        iframe_url = iframe['src']
        if not iframe_url.startswith('http'):
            iframe_url = 'https://www.kaggle.com' + iframe_url
        # 2) Baixar HTML original do iframe
        resp = session.get(iframe_url, timeout=30)
        resp.raise_for_status()
        html_iframe = resp.text
        safe_name = page_url.rstrip('/').split('/')[-1] or 'notebook'
        out_path = RESULTS_DIR / f"{safe_name}_IFRAME.html"
        out_path.write_text(html_iframe, encoding='utf-8')
        saved_paths.append(out_path)
        print(f"  ✓ {len(html_iframe):,} chars → {out_path.name}")
    except Exception as e:
        print(f"  ⚠ {e}")
    time.sleep(1)

print(f"\n✓ Salvos {len(saved_paths)} HTMLs (iframe ou página) em {RESULTS_DIR}.")
# Exemplo do que faltou implementar (celula 0.1): retry em falha, persistir lista saved_paths para célula 4, opção de só iframe (não salvar _page.html).

[1/255] https://www.kaggle.com/code/quangnguynv/xgb-churn-prediction-create-new-features
  ✓ 380,980 chars → xgb-churn-prediction-create-new-features_IFRAME.html
[2/255] https://www.kaggle.com/code/bhaskarmishra44796/ps-s6e3-xgb-baseline
  ✓ 335,615 chars → ps-s6e3-xgb-baseline_IFRAME.html
[3/255] https://www.kaggle.com/code/sehaj1104/s6e3-churn-xgb-hgb-ensemble-v1
  ✓ 304,571 chars → s6e3-churn-xgb-hgb-ensemble-v1_IFRAME.html
[4/255] https://www.kaggle.com/code/larskseme/2026-03-02-lists
  ✓ 304,483 chars → 2026-03-02-lists_IFRAME.html
[5/255] https://www.kaggle.com/code/syedaeman2212/foreign-currency-exchange-rate-analysis
  ✓ 308,717 chars → foreign-currency-exchange-rate-analysis_IFRAME.html
[6/255] https://www.kaggle.com/code/lukhilaksh/global-student-placement-100-beats
  ✓ 356,266 chars → global-student-placement-100-beats_IFRAME.html
[7/255] https://www.kaggle.com/code/saamhm/lightgbm-xgboost-ensemble
  ✓ 405,133 chars → lightgbm-xgboost-ensemble_IFRAME.html
[8/255] https://www

## 4. BeautifulSoup e find_all() — Extração

Passar cada HTML (salvo na célula 3) para **BeautifulSoup** e usar **find_all()** para extrair, em todos os notebooks (ex.: 249), as tags desejadas: `<pre>`, `<table>`, `<img>`, `<svg>`, etc. (conforme celula 0.1: procurar em todos os notebooks as tags).

In [4]:
# Processar todos os HTMLs salvos (iframe) na pasta results/
# Escopo: container do notebook (#site-content div.sc-iKvZkD.cHbUbU) quando existir

def get_notebook_scope(soup):
    """Retorna o escopo para find_all: container do notebook ou o documento inteiro."""
    container = soup.select_one('#site-content div.sc-iKvZkD.cHbUbU')
    if container:
        return container, True
    container = soup.select_one('div.sc-iKvZkD.cHbUbU')
    if container:
        return container, True
    def has_class(classes, *names):
        if not classes:
            return False
        c = classes if isinstance(classes, list) else [classes]
        s = ' '.join(c)
        return all(n in s for n in names)
    container = soup.find('div', class_=lambda c: has_class(c, 'sc-iKvZkD') and (has_class(c, 'cHbUbU') or has_class(c, 'fiBHaZ')))
    if container:
        return container, True
    return soup, False

RESULTS_DIR = Path("results")
html_files = list(RESULTS_DIR.glob("*_IFRAME.html")) or list(RESULTS_DIR.glob("*.html"))
if not html_files:
    html_files = [RESULTS_DIR / "placeholder.html"]
soup = None
pre_tags = []
all_pre_data = []
for html_path in html_files:
    if not html_path.exists():
        continue
    html_content = html_path.read_text(encoding='utf-8')
    soup = BeautifulSoup(html_content, 'html.parser')
    scope, used_container = get_notebook_scope(soup)
    pres = scope.find_all('pre')
    pre_tags.extend(pres)
    all_pre_data.append({'arquivo': html_path.name, 'pre_tags': pres})
print(f"✓ BeautifulSoup: {len(html_files)} arquivo(s), total find_all('pre') = {len(pre_tags)}.")

# --- Exemplo de validação: URL itahiro/deepseek-r1-distill-qwen-7b ---
html_ex = None
EXAMPLE_URL = "https://www.kaggle.com/code/itahiro/deepseek-r1-distill-qwen-7b/notebook"
safe_name = EXAMPLE_URL.rstrip("/").split("/")[-2] if "/notebook" in EXAMPLE_URL else EXAMPLE_URL.rstrip("/").split("/")[-1]
example_path = RESULTS_DIR / f"{safe_name}_IFRAME.html"
if example_path.exists():
    html_ex = example_path.read_text(encoding="utf-8")
    print(f"\n[Validação] Carregado arquivo existente: {example_path.name}")
else:
    print(f"\n[Validação] Baixando iframe para {EXAMPLE_URL} ...")
    html_pagina = fetch_html_with_selenium(EXAMPLE_URL) or ""
    if not html_pagina:
        print("  ⚠ Falha ao obter página.")
    else:
        sp = BeautifulSoup(html_pagina, "html.parser")
        ifr = sp.find("iframe", id="rendered-kernel-content")
        if ifr and ifr.get("src"):
            iframe_url = ifr["src"] if ifr["src"].startswith("http") else "https://www.kaggle.com" + ifr["src"]
            _session = requests.Session()
            _session.headers.update({"User-Agent": "Mozilla/5.0", "Referer": "https://www.kaggle.com/"})
            resp = _session.get(iframe_url, timeout=30)
            resp.raise_for_status()
            html_ex = resp.text
            example_path.parent.mkdir(parents=True, exist_ok=True)
            example_path.write_text(html_ex, encoding="utf-8")
            print(f"  ✓ Iframe salvo: {example_path.name}")
        else:
            html_ex = html_pagina
            print("  ⚠ Iframe não encontrado; usando HTML da página.")
if html_ex:
    soup_ex = BeautifulSoup(html_ex, "html.parser")
    scope_ex, container_found = get_notebook_scope(soup_ex)
    pres_ex = scope_ex.find_all("pre")
    tables_ex = scope_ex.find_all("table")
    imgs_ex = scope_ex.find_all("img")
    print(f"\n[Validação] Container (#site-content div.sc-iKvZkD.cHbUbU ou fallback): {'sim' if container_found else 'não'}")
    print(f"  • <pre>: {len(pres_ex)}  • <table>: {len(tables_ex)}  • <img>: {len(imgs_ex)}")
    for i, pre in enumerate(pres_ex[:3], 1):
        txt = pre.get_text().strip()[:200]
        print(f"  Preview pre #{i}: {txt}...")

✓ BeautifulSoup: 250 arquivo(s), total find_all('pre') = 11568.

[Validação] Carregado arquivo existente: deepseek-r1-distill-qwen-7b_IFRAME.html

[Validação] Container (#site-content div.sc-iKvZkD.cHbUbU ou fallback): não
  • <pre>: 23  • <table>: 0  • <img>: 0
  Preview pre #1: import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRITON_PTXAS_PATH"] = "/usr/local/cuda/bin/ptxas"
import re
import time
import rando...
  Preview pre #2: INFO 03-20 16:11:48 __init__.py:183] Automatically detected platform cuda.
PyTorch version: 2.5.1+cu124
vLLM: 0.7.1...
  Preview pre #3: def seed_everything(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.c...


### Exemplos find_all (opcional)

Outras tags que pode extrair em todos os notebooks: `find_all('a', href=True)`, `find_all('table')`, `find_all('img')`, `find_all('svg')`. Variáveis `pre_tags` e `all_pre_data` já definidas acima.

In [5]:
# pre_tags e all_pre_data já definidos na célula 4 (BeautifulSoup e find_all). Exemplos adicionais:
# links = soup.find_all('a', href=True)
# tabelas = soup.find_all('table')
# imgs = soup.find_all('img')
# svgs = soup.find_all('svg')
print(f"✓ Variáveis disponíveis: pre_tags ({len(pre_tags)} itens), all_pre_data ({len(all_pre_data)} arquivos).")

✓ Variáveis disponíveis: pre_tags (128 itens), all_pre_data (6 arquivos).


## 5. Armazenar dados / manipulação

Armazenar os dados extraídos em lista, DataFrame ou estruturas prontas para ML.

In [5]:
extracted_data = []
idx_global = 0
for item in (all_pre_data if 'all_pre_data' in dir() and all_pre_data else [{'arquivo': 'single', 'pre_tags': pre_tags}]):
    arquivo = item.get('arquivo', '')
    for pre_tag in item.get('pre_tags', []):
        idx_global += 1
        code_text = pre_tag.get_text().strip()
        parent = pre_tag.parent
        parent_classes = ' '.join(parent.get('class', [])) if parent and parent.get('class') else 'N/A'
        extracted_data.append({
            'bloco_numero': idx_global,
            'arquivo': arquivo,
            'conteudo': code_text,
            'tamanho_chars': len(code_text),
            'num_linhas': len(code_text.split('\n')) if code_text else 0,
            'parent_classes': parent_classes,
            'tem_import': 'import' in code_text.lower(),
            'data_extracao': datetime.now().isoformat()
        })

df = pd.DataFrame(extracted_data)
print(f"✓ {len(extracted_data)} registros → DataFrame com {len(df.columns)} colunas.")
df.head()

✓ 11568 registros → DataFrame com 8 colunas.


,bloco_numero,arquivo,conteudo,tamanho_chars,num_linhas,parent_classes,tem_import,data_extracao
0,1,0-91407-catboost-is-all-you-need_IFRAME.html,# This Python 3 environment comes with many he...,930,17,highlight hl-ipython3,True,2026-03-03T03:29:46.502825
1,2,0-91407-catboost-is-all-you-need_IFRAME.html,/kaggle/input/competitions/playground-series-s...,190,3,output_subarea output_stream output_stdout out...,False,2026-03-03T03:29:46.502852
2,3,0-91407-catboost-is-all-you-need_IFRAME.html,import numpy as np\nimport pandas as pd\nimpor...,579,17,highlight hl-ipython3,True,2026-03-03T03:29:46.502942
3,4,0-91407-catboost-is-all-you-need_IFRAME.html,"train = pd.read_csv(""/kaggle/input/competition...",266,3,highlight hl-ipython3,False,2026-03-03T03:29:46.502975
4,5,0-91407-catboost-is-all-you-need_IFRAME.html,"train.drop(""id"", axis=1, inplace=True)\ntest.d...",76,2,highlight hl-ipython3,False,2026-03-03T03:29:46.503005


## 6. Salvar em SQL

Salvar os dados em banco SQL (SQLite) — banco "infinito", sempre append conforme demanda, com estrutura útil para ML.

In [6]:
DB_DIR = Path("results")
DB_DIR.mkdir(exist_ok=True)
db_path = DB_DIR / "kaggle_scraping.db"

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Tabela preparada para crescimento contínuo e uso em ML
cursor.execute('''
CREATE TABLE IF NOT EXISTS scraped_blocks (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    bloco_numero INTEGER,
    arquivo TEXT,
    conteudo TEXT,
    tamanho_chars INTEGER,
    num_linhas INTEGER,
    parent_classes TEXT,
    tem_import BOOLEAN,
    data_extracao TEXT,
    url_fonte TEXT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
''')

# Inserir novos registros (append — banco infinito)
url_fonte = (DISCOVERED_NOTEBOOK_URLS or DIRECT_NOTEBOOK_URLS or [''])[0] if 'DISCOVERED_NOTEBOOK_URLS' in dir() else ''
for row in extracted_data:
    cursor.execute('''
    INSERT INTO scraped_blocks (bloco_numero, arquivo, conteudo, tamanho_chars, num_linhas, parent_classes, tem_import, data_extracao, url_fonte)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', (
        row['bloco_numero'],
        row.get('arquivo', ''),
        row['conteudo'],
        row['tamanho_chars'],
        row['num_linhas'],
        row['parent_classes'],
        row['tem_import'],
        row['data_extracao'],
        url_fonte
    ))

conn.commit()
cursor.execute('SELECT COUNT(*) FROM scraped_blocks')
total = cursor.fetchone()[0]
conn.close()

print(f"✓ Dados salvos em SQLite: {db_path}")
print(f"  Total de registros na tabela: {total}")
print("  (Execute esta célula várias vezes para ir adicionando mais dados.)")

✓ Dados salvos em SQLite: results\kaggle_scraping.db
  Total de registros na tabela: 11696
  (Execute esta célula várias vezes para ir adicionando mais dados.)


In [9]:
# Nova célula: imprimir todo o conteúdo da coluna `conteudo` para um arquivo específico
target_file = "aimo2-starter-llm-code-baseline-lb-2_IFRAME.html"

if 'df' not in globals():
    raise NameError("DataFrame `df` não encontrado. Execute a célula 5 antes desta.")

if 'arquivo' not in df.columns or 'conteudo' not in df.columns:
    raise KeyError("As colunas `arquivo` e/ou `conteudo` não existem em `df`.")

subset = df.loc[df['arquivo'] == target_file, 'conteudo']

print(f"Arquivo alvo: {target_file}")
print(f"Total de registros encontrados: {len(subset)}")

if subset.empty:
    print("Nenhum registro encontrado para este arquivo na coluna `arquivo`.")
else:
    for i, conteudo in enumerate(subset, 1):
        print("\n" + "=" * 100)
        print(f"Bloco {i}/{len(subset)}")
        print("=" * 100)
        print(conteudo)

Arquivo alvo: aimo2-starter-llm-code-baseline-lb-2_IFRAME.html
Total de registros encontrados: 23

Bloco 1/23
import os, re, sys, subprocess, time, gc, math, random
# Notebook start time
T0_NB = time.time()

import numpy as np
from numpy.random import choice
import pandas as pd
import polars as pl
from tqdm import tqdm
from collections import defaultdict, Counter

import torch; torch.backends.cuda.enable_mem_efficient_sdp(False)

import transformers
from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer, 
    AutoConfig,
    StoppingCriteria,
    StoppingCriteriaList,
    set_seed
)

import kaggle_evaluation.aimo_2_inference_server

print(f"Transformers Version: {transformers.__version__}")
set_seed(42)

Bloco 2/23
Transformers Version: 4.39.3

Bloco 3/23
2024-10-24 20:56:26.240187: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-1

In [ ]:
# Nova célula: imprimir tabelas extraídas do arquivo HTML alvo
target_file = 'https://www.kaggle.com/code/quangnguynv/xgb-churn-prediction-create-new-features'
target_path = Path("results") / target_file

if not target_path.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {target_path}")

html_content = target_path.read_text(encoding="utf-8", errors="ignore")
soup_target = BeautifulSoup(html_content, "html.parser")

tables = soup_target.find_all("table")
print(f"Arquivo alvo: {target_file}")
print(f"Total de <table> encontradas via BeautifulSoup: {len(tables)}")

if not tables:
    print("Nenhuma tabela encontrada no HTML.")
else:
    for i, table in enumerate(tables, 1):
        print("\n" + "=" * 100)
        print(f"TABELA HTML {i}/{len(tables)} (texto bruto)")
        print("=" * 100)
        table_text = table.get_text(" ", strip=True)
        print(table_text if table_text else "[Tabela sem texto visível]")

# Tentativa de parse estruturado com pandas
try:
    dfs_tables = pd.read_html(html_content)
    print("\n" + "#" * 100)
    print(f"Total de tabelas parseadas com pandas.read_html: {len(dfs_tables)}")
    print("#" * 100)

    for i, dft in enumerate(dfs_tables, 1):
        print("\n" + "-" * 100)
        print(f"DATAFRAME {i}/{len(dfs_tables)} | shape={dft.shape}")
        print("-" * 100)
        print(dft.to_string(index=False))
except Exception as e:
    print(f"\n⚠ Não foi possível parsear tabelas com pandas.read_html: {e}")

Arquivo alvo: xgb-churn-prediction-create-new-features_IFRAME.html
Total de <table> encontradas via BeautifulSoup: 8

TABELA HTML 1/8 (texto bruto)
id gender SeniorCitizen Partner Dependents tenure PhoneService MultipleLines InternetService OnlineSecurity ... DeviceProtection TechSupport StreamingTV StreamingMovies Contract PaperlessBilling PaymentMethod MonthlyCharges TotalCharges Churn 0 0 Male 0 Yes Yes 29 Yes No DSL Yes ... Yes Yes No No One year Yes Mailed check 60.10 1653.85 No 1 1 Male 0 Yes Yes 58 Yes No DSL Yes ... No Yes Yes No Two year No Credit card (automatic) 69.50 3778.20 No 2 2 Male 0 Yes No 58 Yes Yes Fiber optic No ... No No Yes Yes Month-to-month Yes Electronic check 100.40 5841.35 No 3 3 Female 0 No No 1 Yes No Fiber optic No ... No No No No Month-to-month Yes Electronic check 69.70 70.70 Yes 4 4 Female 0 No No 1 Yes No Fiber optic No ... No No No No Month-to-month Yes Electronic check 70.45 70.45 Yes

TABELA HTML 2/8 (texto bruto)
customerID gender SeniorCitizen Pa

In [ ]:
#criar novo dataframe com resultados do input conteudo,tabelas,imagens... ao aplicar a logica de interpretação npl para extrair informações relevantes para o desafio AIMO, como por exemplo identificar blocos de código que contenham soluções para os problemas propostos, ou tabelas que apresentem resultados de experimentos.ex de soluções para ser considerado:aumento da precisao,diminuir custo,modelo ser bastante generalista,tratar overfittin,super ajuste...,tecnicas de pre processamento que implicaram diretamente no resultado.Identificar a causa do excelente resultado respondendo as perguntas para cada notebook:adaptar pro contextoO que: O que é isso? O que significa? O que causa isso?Por que: Por que isso acontece? Por que é importante?Quem: Quem está envolvido? Quem foi afetado?Qual: Qual é a importância? Qual é o impacto?Quais: Quais são os benefícios? Quais são os desafios?Quanto: Quanto custa? Quanto tempo leva?De que maneira: De que maneira isso funciona? De que maneira isso afeta outras coisas?Quais são as consequências: Quais são as consequências disso? Quais são os resultados esperados?Quais são as alternativas: Quais são as alternativas disponíveis? Quais são as outras opções? Como se compara: Como isso se compara a outras coisas? Como se diferencia?especifico/demonstrativo/direcionado/exemplo/organizado por tópicosprever possíveis problemas apartir de cenários hipotéticosusar limites pra direcionar ajustar demanda(de acordo com o objetivo personalizar parametros IA,temperature,token.....) apartir de comentários caracteristica-causa- inferencias, consecutivas, premissas, silogismo, implicativa(direta e indireta), - caminho inverso(conclusão)-aplicar em outras áreas as proposições sugestões/se fosse uma IA ...OQ PODEMOS APRENDER COM OS ERROS/ACERTOS DO SER HUMANO E OS ERROS/ACERTOS DA IA COM BASE EM TODAS AS PERGUNTAS MONTAMOS CENARIOS HIPOTETICOS PRA  PREVER CAMINHOS CERTOS E ERRADOS(QUANTIFIQUE ATE ONDE ESTA ERRADO E ATE ONDE ESTA CERTO)A seguir, segue uma estrutura geral (template) que você pode adaptar para diferentes conteúdos. Nessa estrutura, incluí "incógnitas" ou marcadores (placeholders) que você deverá substituir com os conceitos e dados específicos do tema tratado. Essa estrutura é flexível e pode ser aplicada a diversos assuntos:ao responder algo simples, tente na imaginação em qual campo pode ser aplicado diretamente e quanto pode ser aplicado diretamente,qual campo pode ser aplicado com adaptações quais as restrições,consequencias direstas e indiretasqual menor tempo pra fazer isso

---

## 1. Introdução- **Contextualização do Tema:**    Apresente brevemente o assunto e a sua relevância.    *Exemplo:* “Hoje, vamos analisar o [TEMA] e entender como [CONCEITO GERAL] influencia [RESULTADO/PROCESSO].”- **Objetivo da Resposta:**    Explique o que se pretende demonstrar ou resolver.    *Exemplo:* “O objetivo é mostrar como [INCÓGNITA_1] se relaciona com [INCÓGNITA_2] para determinar [RESULTADO].”---## 2. Definições e Conceitos Principais- **Definições Básicas:**    Liste as definições essenciais do tema. Utilize incógnitas para representar termos que serão especificados posteriormente.    *Exemplo:*  - [CONCEITO_A]: Definição de [CONCEITO_A] e sua importância.  - [CONCEITO_B]: A definição de [CONCEITO_B] e como ele se conecta a [CONCEITO_A].- **Princípios Fundamentais:**    Explicite os princípios ou fórmulas que regem o tema.    *Exemplo:* “Sabemos que a relação [FÓRMULA]:    \[  X = \sqrt{\frac{Y}{Z}}  \]  mostra como [INCÓGNITA_2] (Y) e [INCÓGNITA_3] (Z) se combinam para definir [RESULTADO].”## 3. Desenvolvimento do Conteúdo- **Divisão em Tópicos:**    Estruture a explicação em seções ou tópicos. Cada tópico deve abordar uma parte do conteúdo    **Tópico 3.1: [subtema/Área 1]**    - **Descrição:** Detalhe o que é [SUBTEMA_1] e sua relação com o tema principal.    - **Incógnitas e Relações:** “Aqui, [CONCEITO_A] influencia [CONCEITO_B] de modo que [RELACIONAMENTO].”  - **Pergunta de Verificação:** “Como [CONCEITO_A] afeta [CONCEITO_B]?”  **Tópico 3.2: [subtema/Área 2]**    - **Descrição:** Explique o [SUBTEMA_2] e demonstre como ele se encaixa no quadro geral.    - **Incógnitas e Relações:** “A partir da relação [FÓRMULA/CONCEITO], se [INCÓGNITA_4] aumentar, então [INCÓGNITA_5] tende a [EFEITO].”  - **Pergunta de Verificação:** “Por que a mudança em [INCÓGNITA_4] impacta [INCÓGNITA_5]?”- **Exemplificação Prática:**   Se aplicável, inclua um exemplo prático para ilustrar as relações.  - *Exemplo:* “Considere dois cenários:     1. Cenário A com [VALOR_1] para [CONCEITO_A] e [VALOR_2] para [CONCEITO_B].      2. Cenário B com [VALOR_3] para [CONCEITO_A] e [VALOR_4] para [CONCEITO_B].    Nesse exemplo, observe que [INCÓGNITA_1] está relacionada a [INCÓGNITA_2] de forma que…”  - **Pergunta de Verificação:** “Quais as principais diferenças entre os Cenários A e B em termos de [CONCEITO_A] e [CONCEITO_B]?”---## 4. Integração e Síntese dos Conceitos- **Integração:**    Conecte os tópicos anteriores para formar uma visão completa do tema.    *Exemplo:* “Vimos que [CONCEITO_A] determina [EFEITO_1] e que [CONCEITO_B] afeta [EFEITO_2]. Dessa forma, o equilíbrio entre [CONCEITO_A] e [CONCEITO_B] leva a [RESULTADO_FINAL].”- **Relação das Incógnitas:**    Explique como as incógnitas se inter-relacionam para responder à questão principal.    *Exemplo:* “Se [INCÓGNITA_1] aumenta, então, conforme a relação [FÓRMULA], [INCÓGNITA_2] diminui, implicando que [CONCLUSÃO].”- **Pergunta Final de Verificação:**    “De que forma a interação entre [CONCEITO_A] e [CONCEITO_B] resulta em [RESULTADO_FINAL]?”## 5. Conclusão- **Resumo dos Pontos Principais:**    Faça uma síntese dos conceitos abordados e das relações estabelecidas.  *Exemplo:* “Resumindo, mostramos que [CONCEITO_A] e [CONCEITO_B] são fundamentais para determinar [RESULTADO FINAL]. A relação entre eles, expressa por [FÓRMULA], demonstra que…”- **Implicações e Aplicações:**    Discuta brevemente as implicações ou aplicações do conteúdo.    *Exemplo:* “Essa compreensão é importante para [APLICAÇÃO/IMPÍCITO], permitindo que possamos [EXEMPLO DE USO PRÁTICO].”- **Pergunta de Reflexão Final:**    “Como você resumiria a importância da interação entre [CONCEITO_A] e [CONCEITO_B] para o tema em discussão?”---## Notas sobre a Flexibilidade e Incógnitas- **Incógnitas:**   Cada marcador como [CONCEITO_A], [INCÓGNITA_1], [FÓRMULA], etc., representa uma parte flexível que depende do conteúdo específico que você está estudando. Ao preparar sua resposta, substitua esses marcadores pelos termos ou valores correspondentes ao tema.  - **Relações:**    Certifique-se de deixar claras as conexões entre as incógnitas. Por exemplo, se INCÓGNITA_1] está diretamente relacionada a [INCÓGNITA_2] via uma fórmula ou um princípio, explique essa relação explicitamente.- **Adaptação:**   Essa estrutura pode ser adaptada para temas de física, biologia, economia, etc. Basta ajustar os conceitos e fórmulas para o contexto específico.